# Common Settings

In [1]:
from transformers.utils import logging

logging.set_verbosity_error()

# Semantic Search with Language Models

## Dense Retrieval

In [2]:
text = """
Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan.
It stars Matthew McConaughey, Anne Hathaway, Jessica Chastain, Bill Irwin, Ellen Burstyn, Matt Damon, and Michael Caine.
Set in a dystopian future where humanity is struggling to survive, the film follows a group of astronauts who travel through a wormhole near Saturn in search of a new home for mankind.

Brothers Christopher and Jonathan Nolan wrote the screenplay, which had its origins in a script Jonathan developed in 2007.
Caltech theoretical physicist and 2017 Nobel laureate in Physics[4] Kip Thorne was an executive producer, acted as a scientific consultant, and wrote a tie-in book, The Science of Interstellar.
Cinematographer Hoyte van Hoytema shot it on 35 mm movie film in the Panavision anamorphic format and IMAX 70 mm.
Principal photography began in late 2013 and took place in Alberta, Iceland, and Los Angeles.
Interstellar uses extensive practical and miniature effects and the company Double Negative created additional digital effects.

Interstellar premiered on October 26, 2014, in Los Angeles.
In the United States, it was first released on film stock, expanding to venues using digital projectors.
The film had a worldwide gross over $677 million (and $773 million with subsequent re-releases), making it the tenth-highest grossing film of 2014.
It received acclaim for its performances, direction, screenplay, musical score, visual effects, ambition, themes, and emotional weight.
It has also received praise from many astronomers for its scientific accuracy and portrayal of theoretical astrophysics. Since its premiere, Interstellar gained a cult following,[5] and now is regarded by many sci-fi experts as one of the best science-fiction films of all time.
Interstellar was nominated for five awards at the 87th Academy Awards, winning Best Visual Effects, and received numerous other accolades.
"""

texts = text.split(".")
texts = [t.strip(" \n") for t in texts if t.strip(" \n")]
text_to_id = {text: idx for idx, text in enumerate(texts)}

print(texts[:3])
print("Number of chunks:", len(texts))


['Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan', 'It stars Matthew McConaughey, Anne Hathaway, Jessica Chastain, Bill Irwin, Ellen Burstyn, Matt Damon, and Michael Caine', 'Set in a dystopian future where humanity is struggling to survive, the film follows a group of astronauts who travel through a wormhole near Saturn in search of a new home for mankind']
Number of chunks: 15


In [3]:
DEFAULT_QUERY = "how precise was the science"

### Dense Retrieval with Cohere API

In [4]:
# %pip install cohere

In [5]:
import cohere

# Paste your API key here. Remember to not share publicly
api_key = ''

# Create and retrieve a Cohere API key from os.cohere.ai
co = cohere.Client(api_key)


In [6]:
import numpy as np

# Get the embeddings
response = co.embed(
    texts=texts,
    model="embed-v4.0",
    input_type="search_document",
    embedding_types=["float"],
)

cohere_embeds = np.asarray(response.embeddings.float, dtype="float32")
print(cohere_embeds.shape)


(15, 1536)


In [7]:
# import faiss

# dim = embeds.shape[1]
# index = faiss.IndexFlatL2(dim)
# index.add(np.float32(embeds))

# We use NumPy for nearest-neighbor search here instead of FAISS.
# This avoids native-library crashes in some notebook environments.
def top_k_indices(values, k, *, largest):
    values = np.asarray(values)
    k = min(k, len(values))
    if k <= 0:
        return np.array([], dtype=int)

    if largest:
        candidate_ids = np.argpartition(values, -k)[-k:]
        return candidate_ids[np.argsort(values[candidate_ids])[::-1]]

    candidate_ids = np.argpartition(values, k - 1)[:k]
    return candidate_ids[np.argsort(values[candidate_ids])]


In [8]:
import pandas as pd


def search(query, number_of_results=3):
    # 1. Get the query's embedding
    query_response = co.embed(
        texts=[query],
        model="embed-v4.0",
        input_type="search_query",
        embedding_types=["float"],
    )
    query_embed = np.asarray(query_response.embeddings.float, dtype="float32")[0]

    # 2. Retrieve the nearest neighbors with squared L2 distance
    # distances , similar_item_ids = index.search(np.float32([query_embed]), number_of_results)
    squared_l2_distances = np.sum((cohere_embeds - query_embed) ** 2, axis=1)
    similar_item_ids = top_k_indices(squared_l2_distances, number_of_results, largest=False)

    # 3. Format the results
    texts_np = np.array(texts)
    results = pd.DataFrame({
        'texts': texts_np[similar_item_ids],
        'distance': squared_l2_distances[similar_item_ids],
    })

    # 4. Print and return the results
    print(f"Query:'{query}'\nNearest neighbors:")
    return results


In [9]:
results = search(query=DEFAULT_QUERY)
results


Query:'how precise was the science'
Nearest neighbors:


,texts,distance
0,It has also received praise from many astronom...,1.293341
1,Caltech theoretical physicist and 2017 Nobel l...,1.564170
2,"Since its premiere, Interstellar gained a cult...",1.630263


### Dense Retrieval with local LLM

In [10]:
from sentence_transformers import SentenceTransformer
import numpy as np
import pandas as pd


def encode_texts(model_name, texts, *, document_prefix=""):
    model = SentenceTransformer(model_name)
    prepared_texts = [f"{document_prefix}{text}" for text in texts]
    embeds = model.encode(prepared_texts, normalize_embeddings=True)
    embeds = np.asarray(embeds, dtype="float32")

    print(f"Embedding:		{embeds.shape}")
    return model, embeds


def build_index(embeds):
    # Normalized embeddings can be searched with a simple inner product.
    return np.asarray(embeds, dtype="float32")


def search_with_model(query, model, index, texts, number_of_results=3, *, query_prefix=""):
    query_embed = model.encode([f"{query_prefix}{query}"], normalize_embeddings=True)
    query_embed = np.asarray(query_embed, dtype="float32")[0]

    scores = index @ query_embed
    similar_item_ids = top_k_indices(scores, number_of_results, largest=True)
    texts_np = np.array(texts)
    results = pd.DataFrame({
        "texts": texts_np[similar_item_ids],
        "score": scores[similar_item_ids]
    })

    print(f"Query:			'{query}'\nNearest neighbors:")
    return results


def run_retrieval_demo(model_name, texts, *, query=DEFAULT_QUERY, number_of_results=3, document_prefix="", query_prefix=""):
    model, embeds = encode_texts(model_name, texts, document_prefix=document_prefix)
    index = build_index(embeds)
    results = search_with_model(
        query,
        model,
        index,
        texts,
        number_of_results=number_of_results,
        query_prefix=query_prefix,
    )
    return model, embeds, index, results


#### all-MiniLM-L6-v2

In [11]:
model, embeds, index, results = run_retrieval_demo(
    "all-MiniLM-L6-v2",
    texts,
)
results


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding:		(15, 384)
Query:			'how precise was the science'
Nearest neighbors:


,texts,score
0,It has also received praise from many astronom...,0.475606
1,Caltech theoretical physicist and 2017 Nobel l...,0.333492
2,Interstellar uses extensive practical and mini...,0.209006


#### sentence-transformers/multi-qa-mpnet-base-cos-v1

In [12]:
model, embeds, index, results = run_retrieval_demo(
    "sentence-transformers/multi-qa-mpnet-base-cos-v1",
    texts,
)
results


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding:		(15, 768)
Query:			'how precise was the science'
Nearest neighbors:


,texts,score
0,It has also received praise from many astronom...,0.376703
1,"It received acclaim for its performances, dire...",0.233058
2,Cinematographer Hoyte van Hoytema shot it on 3...,0.189781


#### intfloat/e5-base-v2

In [13]:
model, embeds, index, results = run_retrieval_demo(
    "intfloat/e5-base-v2",
    texts,
    document_prefix="passage: ",
    query_prefix="query: ",
)
results


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding:		(15, 768)
Query:			'how precise was the science'
Nearest neighbors:


,texts,score
0,It has also received praise from many astronom...,0.787450
1,Caltech theoretical physicist and 2017 Nobel l...,0.767098
2,Cinematographer Hoyte van Hoytema shot it on 3...,0.746845


#### BAAI/bge-base-en-v1.5

In [14]:
model, embeds, index, results = run_retrieval_demo(
    "BAAI/bge-base-en-v1.5",
    texts,
    query_prefix="Represent this sentence for searching relevant passages: ",
)
results


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding:		(15, 768)
Query:			'how precise was the science'
Nearest neighbors:


,texts,score
0,It has also received praise from many astronom...,0.560431
1,Caltech theoretical physicist and 2017 Nobel l...,0.477876
2,"Since its premiere, Interstellar gained a cult...",0.405915


### Keyword Search

In [15]:
# %pip install rank-bm25

In [16]:
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction import _stop_words
import string

def bm25_tokenizer(text):
    tokenized_doc = []
    for token in text.lower().split():
        token = token.strip(string.punctuation)

        if len(token) > 0 and token not in _stop_words.ENGLISH_STOP_WORDS:
            tokenized_doc.append(token)
    return tokenized_doc


In [17]:
from tqdm import tqdm

tokenized_corpus = []
for passage in tqdm(texts):
    tokenized_corpus.append(bm25_tokenizer(passage))

bm25 = BM25Okapi(tokenized_corpus)


100%|████████████████████████████████████████████████████████████████████████████████████████| 15/15 [00:00<00:00, 194781.92it/s]


In [18]:
def bm25_retrieve(query, num_candidates=15):
    bm25_scores = bm25.get_scores(bm25_tokenizer(query))
    corpus_size = len(bm25_scores)
    if corpus_size == 0:
        return []

    num_candidates = min(num_candidates, corpus_size)
    top_n = np.argpartition(bm25_scores, -num_candidates)[-num_candidates:]
    bm25_hits = [{"corpus_id": idx, "score": float(bm25_scores[idx])} for idx in top_n]
    return sorted(bm25_hits, key=lambda x: x["score"], reverse=True)


def print_hits(hits, texts, *, score_key, top_k=3, title):
    print(title)
    for hit in hits[:top_k]:
        print("\t{:.6f}\t{}".format(
            hit[score_key],
            texts[hit["corpus_id"]].replace("\n", " ")
        ))


def keyword_search(query, top_k=3, num_candidates=15):
    print("Input question:", query)

    bm25_hits = bm25_retrieve(query, num_candidates=num_candidates)
    top_k = min(top_k, len(bm25_hits))
    if top_k == 0:
        print("Corpus is empty.")
        return []

    print_hits(
        bm25_hits,
        texts,
        score_key="score",
        top_k=top_k,
        title=f"Top-{top_k} lexical search (BM25) hits",
    )
    return bm25_hits[:top_k]


In [19]:
keyword_search(query = DEFAULT_QUERY, num_candidates=14)


Input question: how precise was the science
Top-3 lexical search (BM25) hits
	1.788605	Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan
	1.372650	Caltech theoretical physicist and 2017 Nobel laureate in Physics[4] Kip Thorne was an executive producer, acted as a scientific consultant, and wrote a tie-in book, The Science of Interstellar
	0.000000	Set in a dystopian future where humanity is struggling to survive, the film follows a group of astronauts who travel through a wormhole near Saturn in search of a new home for mankind


[{'corpus_id': np.int64(0), 'score': 1.788604950756303},
 {'corpus_id': np.int64(4), 'score': 1.372650311045535},
 {'corpus_id': np.int64(2), 'score': 0.0}]

### Caveats of Dense Retrieval

In [20]:
model, embeds, index, results = run_retrieval_demo(
    "all-MiniLM-L6-v2",
    texts,
    query="What is the mass of the moon?"
)
results


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding:		(15, 384)
Query:			'What is the mass of the moon?'
Nearest neighbors:


,texts,score
0,It has also received praise from many astronom...,0.246421
1,Cinematographer Hoyte van Hoytema shot it on 3...,0.191763
2,The film had a worldwide gross over $677 milli...,0.165496


## Reranking

### Cohere API

In [21]:
results = co.rerank(query=DEFAULT_QUERY, documents=texts, top_n=3, return_documents=True)
results.results


[RerankResponseResultsItem(document=RerankResponseResultsItemDocument(text='It has also received praise from many astronomers for its scientific accuracy and portrayal of theoretical astrophysics'), index=12, relevance_score=0.15239799),
 RerankResponseResultsItem(document=RerankResponseResultsItemDocument(text='The film had a worldwide gross over $677 million (and $773 million with subsequent re-releases), making it the tenth-highest grossing film of 2014'), index=10, relevance_score=0.050354082),
 RerankResponseResultsItem(document=RerankResponseResultsItemDocument(text='Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan'), index=0, relevance_score=0.0350424)]

In [22]:
for idx, result in enumerate(results.results):
    print(idx, result.relevance_score , result.document.text)


0 0.15239799 It has also received praise from many astronomers for its scientific accuracy and portrayal of theoretical astrophysics
1 0.050354082 The film had a worldwide gross over $677 million (and $773 million with subsequent re-releases), making it the tenth-highest grossing film of 2014
2 0.0350424 Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan


In [23]:
def keyword_and_cohere_reranking_search(query, top_k=3, num_candidates=10):
    print("Input question:", query)

    ##### BM25 search (lexical search) #####
    bm25_scores = bm25.get_scores(bm25_tokenizer(query))
    top_n = np.argpartition(bm25_scores, -num_candidates)[-num_candidates:]
    bm25_hits = [{'corpus_id': idx, 'score': bm25_scores[idx]} for idx in top_n]
    bm25_hits = sorted(bm25_hits, key=lambda x: x['score'], reverse=True)

    print(f"Top-3 lexical search (BM25) hits")
    for hit in bm25_hits[0:top_k]:
        print("\t{:.3f}\t{}".format(hit['score'], texts[hit['corpus_id']].replace("\n", " ")))

    # Add re-ranking
    docs = [texts[hit['corpus_id']] for hit in bm25_hits]

    print(f"\nTop-3 hits by rank-API ({len(bm25_hits)} BM25 hits re-ranked)")
    results = co.rerank(query=query, documents=docs, top_n=top_k, return_documents=True)
    for hit in results.results:
        print("\t{:.3f}\t{}".format(hit.relevance_score, hit.document.text.replace("\n", " ")))


In [24]:
keyword_and_cohere_reranking_search(query=DEFAULT_QUERY)


Input question: how precise was the science
Top-3 lexical search (BM25) hits
	1.789	Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan
	1.373	Caltech theoretical physicist and 2017 Nobel laureate in Physics[4] Kip Thorne was an executive producer, acted as a scientific consultant, and wrote a tie-in book, The Science of Interstellar
	0.000	Interstellar uses extensive practical and miniature effects and the company Double Negative created additional digital effects

Top-3 hits by rank-API (10 BM25 hits re-ranked)
	0.035	Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan
	0.032	It stars Matthew McConaughey, Anne Hathaway, Jessica Chastain, Bill Irwin, Ellen Burstyn, Matt Damon, and Michael Caine
	0.031	Caltech theoretical physicist and 2017 Nobel laureate in Physics[4] Kip Thorne was an executive producer, acted as a scientific consultant, and wrote a tie-in book, The Science of In

### cross-encoder/ms-marco-MiniLM-L-6-v2

In [25]:
import torch
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2",
    activation_fn=torch.nn.Sigmoid()
)


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [26]:
def dense_results_to_hits(results, text_to_id):
    return [
        {
            "corpus_id": text_to_id[row["texts"]],
            "score": float(row["score"]),
        }
        for _, row in results.iterrows()
    ]


def rerank_hits(query, hits, texts, reranker, top_k=3):
    """
    hits: the first stage of the search pipeline, e.g.:
        [{"corpus_id": 5, "score": 1.23}, {"corpus_id": 2, "score": 0.88}]
    texts: raw text
    """
    candidate_docs = [texts[hit["corpus_id"]] for hit in hits]
    pairs = [[query, doc] for doc in candidate_docs]
    rerank_scores = reranker.predict(pairs)

    reranked = []
    for hit, doc, rerank_score in zip(hits, candidate_docs, rerank_scores):
        reranked.append({
            "corpus_id": hit["corpus_id"],
            "first_stage_score": float(hit["score"]),
            "rerank_score": float(rerank_score),
            "text": doc,
        })

    reranked = sorted(reranked, key=lambda x: x["rerank_score"], reverse=True)
    return reranked[:top_k]


In [27]:
# 1) dense retrieval
model, embeds, index, results = run_retrieval_demo(
    "all-MiniLM-L6-v2",
    texts,
    query=DEFAULT_QUERY,
    number_of_results=12
)
display(results)

# 2) convert to required format
dense_hits = dense_results_to_hits(results, text_to_id)

# 3) rerank via cross-encoder
reranked_top3 = pd.DataFrame(
    rerank_hits(
        query=DEFAULT_QUERY,
        hits=dense_hits,
        texts=texts,
        reranker=reranker,
    )
).reset_index(drop=True)

print("Reranked results:")
display(reranked_top3)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding:		(15, 384)
Query:			'how precise was the science'
Nearest neighbors:


,texts,score
0,It has also received praise from many astronom...,0.475606
1,Caltech theoretical physicist and 2017 Nobel l...,0.333492
2,Interstellar uses extensive practical and mini...,0.209006
3,"Since its premiere, Interstellar gained a cult...",0.207521
4,Cinematographer Hoyte van Hoytema shot it on 3...,0.143769
5,Interstellar is a 2014 epic science fiction fi...,0.097521
6,"It received acclaim for its performances, dire...",0.091610
7,Interstellar was nominated for five awards at ...,0.090975
8,"Interstellar premiered on October 26, 2014, in...",0.062359
9,Principal photography began in late 2013 and t...,0.058045


Reranked results:


,corpus_id,first_stage_score,rerank_score,text
0,12,0.475606,0.000046,It has also received praise from many astronom...
1,4,0.333492,0.000017,Caltech theoretical physicist and 2017 Nobel l...
2,0,0.097521,0.000014,Interstellar is a 2014 epic science fiction fi...


In [28]:
for idx, row in reranked_top3.iterrows():
    print(f"{idx}. rerank_score={row['rerank_score']:.6f}")
    print(row["text"])
    print("-" * 60)


0. rerank_score=0.000046
It has also received praise from many astronomers for its scientific accuracy and portrayal of theoretical astrophysics
------------------------------------------------------------
1. rerank_score=0.000017
Caltech theoretical physicist and 2017 Nobel laureate in Physics[4] Kip Thorne was an executive producer, acted as a scientific consultant, and wrote a tie-in book, The Science of Interstellar
------------------------------------------------------------
2. rerank_score=0.000014
Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan
------------------------------------------------------------


In [29]:
def keyword_and_reranking_search(query, top_k=3, num_candidates=10):
    print("Input question:", query)

    bm25_hits = bm25_retrieve(query, num_candidates=num_candidates)
    top_k = min(top_k, len(bm25_hits))
    if top_k == 0:
        print("Corpus is empty.")
        return []

    print_hits(
        bm25_hits,
        texts,
        score_key="score",
        top_k=top_k,
        title=f"Top-{top_k} lexical search (BM25) hits",
    )

    reranked_hits = rerank_hits(
        query=query,
        hits=bm25_hits,
        texts=texts,
        reranker=reranker,
        top_k=top_k,
    )

    print_hits(
        reranked_hits,
        texts,
        score_key="rerank_score",
        top_k=top_k,
        title=f"\nTop-{top_k} hits after reranking ({len(bm25_hits)} BM25 hits reranked)",
    )

    return reranked_hits


In [30]:
keyword_and_reranking_search(query = DEFAULT_QUERY)


Input question: how precise was the science
Top-3 lexical search (BM25) hits
	1.788605	Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan
	1.372650	Caltech theoretical physicist and 2017 Nobel laureate in Physics[4] Kip Thorne was an executive producer, acted as a scientific consultant, and wrote a tie-in book, The Science of Interstellar
	0.000000	Interstellar uses extensive practical and miniature effects and the company Double Negative created additional digital effects

Top-3 hits after reranking (10 BM25 hits reranked)
	0.000017	Caltech theoretical physicist and 2017 Nobel laureate in Physics[4] Kip Thorne was an executive producer, acted as a scientific consultant, and wrote a tie-in book, The Science of Interstellar
	0.000014	Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan
	0.000013	It stars Matthew McConaughey, Anne Hathaway, Jessica Chastain, Bill Irwin, Ellen Burstyn

[{'corpus_id': np.int64(4),
  'first_stage_score': 1.372650311045535,
  'rerank_score': 1.6586092897341587e-05,
  'text': 'Caltech theoretical physicist and 2017 Nobel laureate in Physics[4] Kip Thorne was an executive producer, acted as a scientific consultant, and wrote a tie-in book, The Science of Interstellar'},
 {'corpus_id': np.int64(0),
  'first_stage_score': 1.788604950756303,
  'rerank_score': 1.3945723367214669e-05,
  'text': 'Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan'},
 {'corpus_id': np.int64(1),
  'first_stage_score': 0.0,
  'rerank_score': 1.2732823961414397e-05,
  'text': 'It stars Matthew McConaughey, Anne Hathaway, Jessica Chastain, Bill Irwin, Ellen Burstyn, Matt Damon, and Michael Caine'}]

# Retrieval-Augmented Generation

## Example: Grounded Generation with an LLM API

In [31]:
query = "income generated"

# 1- Retrieval
# We'll use embedding search. But ideally we'd do hybrid
results = search(query)
print(results)

# 2- Grounded Generation
docs_dict = [{'text': text} for text in results['texts']]
response = co.chat(
    message = query,
    documents=docs_dict
)

print("\n"+response.text)


Query:'income generated'
Nearest neighbors:
                                               texts  distance
0  The film had a worldwide gross over $677 milli...  1.684728
1  Brothers Christopher and Jonathan Nolan wrote ...  1.688337
2  Interstellar is a 2014 epic science fiction fi...  1.708297

The film Interstellar had a worldwide gross of over $677 million, and $773 million with subsequent re-releases, making it the tenth-highest grossing film of 2014.


In [32]:
response


NonStreamedChatResponse(text='The film Interstellar had a worldwide gross of over $677 million, and $773 million with subsequent re-releases, making it the tenth-highest grossing film of 2014.', generation_id='0fba85bb-01f1-427d-b7aa-ae6d30ccbef0', response_id='6f662462-8a68-4145-9af1-37e83d9eb7f7', citations=[ChatCitation(start=9, end=21, text='Interstellar', document_ids=['doc_2'], type='TEXT_CONTENT'), ChatCitation(start=28, end=64, text='worldwide gross of over $677 million', document_ids=['doc_0'], type='TEXT_CONTENT'), ChatCitation(start=70, end=110, text='$773 million with subsequent re-releases', document_ids=['doc_0'], type='TEXT_CONTENT'), ChatCitation(start=126, end=162, text='tenth-highest grossing film of 2014.', document_ids=['doc_0'], type='TEXT_CONTENT')], documents=[{'id': 'doc_2', 'text': 'Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan'}, {'id': 'doc_0', 'text': 'The film had a worldwide gross over $677 million

In [33]:
response.citations


[ChatCitation(start=9, end=21, text='Interstellar', document_ids=['doc_2'], type='TEXT_CONTENT'),
 ChatCitation(start=28, end=64, text='worldwide gross of over $677 million', document_ids=['doc_0'], type='TEXT_CONTENT'),
 ChatCitation(start=70, end=110, text='$773 million with subsequent re-releases', document_ids=['doc_0'], type='TEXT_CONTENT'),
 ChatCitation(start=126, end=162, text='tenth-highest grossing film of 2014.', document_ids=['doc_0'], type='TEXT_CONTENT')]

## Example: RAG with Local Models

### Loading the Generation Model

In [34]:
# from langchain import LlamaCpp

# # Make sure the model path is correct for your system!
# llm = LlamaCpp(
#     model_path="Phi-3-mini-4k-instruct-q4.gguf",
#     n_gpu_layers=-1,
#     max_tokens=500,
#     n_ctx=2048,
#     seed=42,
#     verbose=False
# )

from pathlib import Path
from langchain_community.llms import LlamaCpp

model_path = Path.home() / ".cache" / "huggingface" / "hub" / \
    "models--microsoft--Phi-3-mini-4k-instruct-gguf" / "snapshots" / \
    "a64113399c2f6b8ad3e11c394733a2ddadaa7f33" / "Phi-3-mini-4k-instruct-fp16.gguf"

print(model_path)
print(model_path.exists())

llm = LlamaCpp(
    model_path=str(model_path),
    n_gpu_layers=-1,
    max_tokens=500,
    n_ctx=2048,
    seed=42,
    verbose=False
)


/Users/jenslewie/.cache/huggingface/hub/models--microsoft--Phi-3-mini-4k-instruct-gguf/snapshots/a64113399c2f6b8ad3e11c394733a2ddadaa7f33/Phi-3-mini-4k-instruct-fp16.gguf
True


llama_context: n_ctx_seq (2048) < n_ctx_train (4096) -- the full capacity of the model will not be utilized


### Loading the Embedding Model

In [35]:
# %pip install langchain-huggingface

In [36]:
# from langchain.embeddings.huggingface import HuggingFaceEmbeddings

# # Embedding Model for converting text to numerical representations
# embedding_model = HuggingFaceEmbeddings(
#     model_name='BAAI/bge-small-en-v1.5'
# )

from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5"
)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

### Preparing the Vector Database

In [37]:
# from langchain.vectorstores import FAISS
from langchain_community.vectorstores import FAISS

# Create a local vector database
db = FAISS.from_texts(texts, embedding_model)


### The RAG Prompt

In [38]:
# from langchain import PromptTemplate
# from langchain.chains import RetrievalQA
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import RetrievalQA


# Create a prompt template
template = """<|user|>
Relevant information:
{context}

Provide a concise answer the following question using the relevant information provided above:
{question}<|end|>
<|assistant|>"""

prompt = PromptTemplate(
    template=template,
    input_variables=["context", "question"]
)

# RAG Pipeline
rag = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type='stuff',
    retriever=db.as_retriever(),
    chain_type_kwargs={
        "prompt": prompt
    },
    verbose=True
)


In [39]:
rag.invoke('Income generated')



> Entering new RetrievalQA chain...

> Finished chain.


{'query': 'Income generated',
 'result': ' The film generated over $677 million worldwide from its initial release, with an additional $773 million following subsequent re-releases. It was also first released in the United States on film stock and later expanded to digital projectors in venues.'}